<a href="https://colab.research.google.com/github/prjhseo/movieLens_recommendation/blob/main/MovieLens_item_based_cf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MovieLens 1M Dataset 분석


## DataSet 파일 설명
RATINGS FILE DESCRIPTION
================================================================================

All ratings are contained in the file "ratings.dat" and are in the
following format:

UserID::MovieID::Rating::Timestamp

- UserIDs range between 1 and 6040
- MovieIDs range between 1 and 3952
- Ratings are made on a 5-star scale (whole-star ratings only)
- Timestamp is represented in seconds since the epoch as returned by time(2)
- Each user has at least 20 ratings

USERS FILE DESCRIPTION
================================================================================

User information is in the file "users.dat" and is in the following
format:

UserID::Gender::Age::Occupation::Zip-code

All demographic information is provided voluntarily by the users and is
not checked for accuracy.  Only users who have provided some demographic
information are included in this data set.

- Gender is denoted by a "M" for male and "F" for female
- Age is chosen from the following ranges:

	*  1:  "Under 18"
	* 18:  "18-24"
	* 25:  "25-34"
	* 35:  "35-44"
	* 45:  "45-49"
	* 50:  "50-55"
	* 56:  "56+"

- Occupation is chosen from the following choices:

	*  0:  "other" or not specified
	*  1:  "academic/educator"
	*  2:  "artist"
	*  3:  "clerical/admin"
	*  4:  "college/grad student"
	*  5:  "customer service"
	*  6:  "doctor/health care"
	*  7:  "executive/managerial"
	*  8:  "farmer"
	*  9:  "homemaker"
	* 10:  "K-12 student"
	* 11:  "lawyer"
	* 12:  "programmer"
	* 13:  "retired"
	* 14:  "sales/marketing"
	* 15:  "scientist"
	* 16:  "self-employed"
	* 17:  "technician/engineer"
	* 18:  "tradesman/craftsman"
	* 19:  "unemployed"
	* 20:  "writer"

MOVIES FILE DESCRIPTION
================================================================================

Movie information is in the file "movies.dat" and is in the following
format:

MovieID::Title::Genres

- Titles are identical to titles provided by the IMDB (including
year of release)
- Genres are pipe-separated and are selected from the following genres:

	* Action
	* Adventure
	* Animation
	* Children's
	* Comedy
	* Crime
	* Documentary
	* Drama
	* Fantasy
	* Film-Noir
	* Horror
	* Musical
	* Mystery
	* Romance
	* Sci-Fi
	* Thriller
	* War
	* Western

- Some MovieIDs do not correspond to a movie due to accidental duplicate
entries and/or test entries
- Movies are mostly entered by hand, so errors and inconsistencies may exist


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

%matplotlib inline

# 데이터 로드

In [57]:
path = r"/content/drive/MyDrive/데이터분석/ml-1m/"

movies = pd.read_csv(
    path + "movies.dat",
    sep="::",
    engine="python",
    header=None,
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1"
)

print(movies.head())
print(movies.shape)

   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
(3883, 3)


In [58]:
ratings = pd.read_csv(
    path + "ratings.dat",
    sep="::",
    engine="python",
    header=None,
    names=["UserID","MovieID","Rating","Timestamp"]
)

users = pd.read_csv(
    path + "users.dat",
    sep="::",
    engine="python",
    header=None,
    names=["UserID","Gender","Age","Occupation","Zipcode"]
)

print(users.head())
print(users.shape)
print(ratings.head())
print(ratings.shape)

   UserID Gender  Age  Occupation Zipcode
0       1      F    1          10   48067
1       2      M   56          16   70072
2       3      M   25          15   55117
3       4      M   45           7   02460
4       5      M   25          20   55455
(6040, 5)
   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
(1000209, 4)


# Timestamp 기반 split

In [59]:
# ratings 시간 순 정렬
ratings = ratings.sort_values(
    ['UserID','Timestamp']
)

In [60]:
ratings.head()

,UserID,MovieID,Rating,Timestamp
31,1,3186,4,978300019
22,1,1270,5,978300055
27,1,1721,4,978300055
37,1,1022,5,978300055
24,1,2340,3,978300103


In [61]:
test = ratings.groupby(
    'UserID'
).tail(1)

In [62]:
test.head()

,UserID,MovieID,Rating,Timestamp
25,1,48,5,978824351
136,2,1917,3,978300174
232,3,2081,4,978298504
243,4,1954,5,978294282
258,5,288,2,978246585


In [63]:
train=ratings.drop(test.index)

In [64]:
print("Train shape:",train.shape)
print("Test shape:",test.shape)

Train shape: (994169, 4)
Test shape: (6040, 4)


# Train matrix 생성

In [65]:
train_movie_rating=pd.merge(train,movies,on='MovieID')
train_movie_rating.head()

,UserID,MovieID,Rating,Timestamp,Title,Genres
0,1,3186,4,978300019,"Girl, Interrupted (1999)",Drama
1,1,1270,5,978300055,Back to the Future (1985),Comedy|Sci-Fi
2,1,1721,4,978300055,Titanic (1997),Drama|Romance
3,1,1022,5,978300055,Cinderella (1950),Animation|Children's|Musical
4,1,2340,3,978300103,Meet Joe Black (1998),Romance


In [66]:
train_matrix=train_movie_rating.pivot_table('Rating',index='Title',columns='UserID')
train_matrix.head()

UserID,1,2,3,4,5,6,7,8,9,10,...,6031,6032,6033,6034,6035,6036,6037,6038,6039,6040
Title,,,,,,,,,,,,,,,,,,,,,
"$1,000,000 Duck (1971)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Night Mother (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"'burbs, The (1989)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...And Justice for All (1979),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#  item similarity 계산

In [67]:
item_similarity = train_matrix.T.corr(
    method='pearson'
).fillna(0)

In [68]:
print(train_matrix.shape)
print(item_similarity.shape)

(3704, 6040)
(3704, 3704)


In [69]:
item_similarity = pd.DataFrame(data = item_similarity, index = train_matrix.index, columns = train_matrix.index)

In [70]:

item_similarity.head()


Title,"$1,000,000 Duck (1971)",'Night Mother (1986),'Til There Was You (1997),"'burbs, The (1989)",...And Justice for All (1979),1-900 (1994),10 Things I Hate About You (1999),101 Dalmatians (1961),101 Dalmatians (1996),12 Angry Men (1957),...,"Young Poisoner's Handbook, The (1995)",Young Sherlock Holmes (1985),Young and Innocent (1937),Your Friends and Neighbors (1998),Zachariah (1971),"Zed & Two Noughts, A (1985)",Zero Effect (1998),Zero Kelvin (Kjærlighetens kjøtere) (1995),Zeus and Roxanne (1997),eXistenZ (1999)
Title,,,,,,,,,,,,,,,,,,,,,
"$1,000,000 Duck (1971)",1.000000e+00,0.522233,0.000000,-9.410021e-18,0.422577,0.0,-0.449490,0.561074,0.150584,0.125209,...,1.000000,-0.106243,0.000000,0.000000,0.0,-1.000000,0.000000,0.0,0.944911,-0.500000
'Night Mother (1986),5.222330e-01,1.000000,-0.177705,2.400000e-01,0.216272,0.0,0.266158,0.220347,-0.199239,0.284342,...,0.774597,-0.276359,1.000000,-0.052758,0.0,-0.492518,0.082580,0.0,0.000000,0.100372
'Til There Was You (1997),0.000000e+00,-0.177705,1.000000,7.126268e-01,0.719676,0.0,0.300273,0.457984,0.564288,0.210259,...,0.000000,-0.016137,0.000000,0.059235,0.0,0.000000,0.221595,0.0,1.000000,0.303731
"'burbs, The (1989)",-9.410021e-18,0.240000,0.712627,1.000000e+00,0.089727,0.0,0.189689,0.088541,0.101064,0.038153,...,0.355654,0.288760,0.000000,0.158997,0.0,0.080064,0.039503,0.0,0.000000,0.135172
...And Justice for All (1979),4.225771e-01,0.216272,0.719676,8.972749e-02,1.000000,0.0,0.291124,0.166965,-0.142597,0.291794,...,-0.279257,0.062035,0.310087,0.159503,0.0,-0.542037,0.200237,0.0,0.000000,-0.024037


# 추천함수

In [71]:
# 유사한 영화 추천
def get_item_based_cf(title):
    return item_similarity[title].sort_values(ascending=False)[:6]

In [72]:
get_item_based_cf('Godfather, The (1972)')

,"Godfather, The (1972)"
Title,
Heaven (1998),1.0
Oxygen (1999),1.0
"Love Bewitched, A (El Amor Brujo) (1986)",1.0
Brothers in Trouble (1995),1.0
Better Living (1998),1.0
"Sticky Fingers of Time, The (1997)",1.0


# 평점 예측함수

In [76]:
def predict_rating(user_id,title):

    # 1. Check if user_id and title exist in the training data
    if user_id not in train_matrix.columns:
        return np.nan

    if title not in item_similarity.columns:
        return np.nan

    # 2. Get all movies rated by the user
    user_rated_movies = train_matrix[user_id].dropna()

    # 3. If the user hasn't rated any movies, return NaN
    if user_rated_movies.empty:
        return np.nan

    # 4. Get the similarity scores for the target movie with all other movies
    #    Exclude the similarity of the movie with itself
    all_similarities = item_similarity[title].drop(title, errors='ignore')

    # 5. Find common movies: those rated by the user AND have a similarity score with the target movie
    common_movies = user_rated_movies.index.intersection(all_similarities.index)

    # 6. If there are no common movies, return NaN
    if common_movies.empty:
        return np.nan

    # 7. Filter both similarity scores and user ratings to include only common movies
    filtered_similarities = all_similarities.loc[common_movies]
    filtered_user_ratings = user_rated_movies.loc[common_movies]

    # 8. Calculate the denominator (sum of absolute similarities of common movies)
    denominator = np.sum(np.abs(filtered_similarities))

    # 9. If denominator is zero, return NaN to avoid division by zero
    if denominator == 0:
        return np.nan

    # 10. Calculate the numerator (dot product of filtered similarities and filtered user ratings)
    prediction = np.dot(filtered_similarities, filtered_user_ratings) / denominator

    return prediction

# test 평가

In [77]:
predictions=[]
actuals=[]

for row in test.itertuples():

    try:

        title = movies.loc[
            movies['MovieID']==row.MovieID,
            'Title'
        ].values[0]

        pred = predict_rating(
            row.UserID,
            title
        )

        if not np.isnan(pred):

            predictions.append(pred)
            actuals.append(row.Rating)

    except:
        continue

# RMSE 계산

In [78]:
rmse = np.sqrt(
    mean_squared_error(
        actuals,
        predictions
    )
)

print("RMSE:", rmse)

RMSE: 1.2633751804027327


코사인 유사도 : RMSE: 3.5304094047380894  
피어슨 fillna(0) 미리 RMSE: 3.5464852343240887  